# Experiment 3 - Prompt Engineering using Large Language Models


In [ ]:
!pip -q install transformers sentencepiece torch pandas

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch, re, textwrap, time

MODEL_NAME = "google/flan-t5-small"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model.eval()
print("Model:", MODEL_NAME, "| device:", device)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model: google/flan-t5-small | device: cpu


In [ ]:
def generate(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=2)
    return tokenizer.decode(out[0], skip_special_tokens=True)

tasks = [
    {
        "task": "Explain a concept",
        "input": "Explain why a cricket batter should keep their head balanced while playing a straight drive.",
        "few": "Input: Explain why a batter watches the ball.\nOutput: Watching the ball helps the batter judge its line and length and time the shot.\n\nInput: Explain why footwork matters.\nOutput: Footwork helps the batter reach the ball while maintaining balance.\n\nInput: " + "Explain why a cricket batter should keep their head balanced while playing a straight drive.",
    },
    {
        "task": "Create a drill",
        "input": "Suggest a safe 10-minute beginner drill to practise front-foot defence.",
        "few": "Input: Give a 5-minute warm-up.\nOutput: 1) Easy jogging, 2) shoulder mobility, 3) gentle dynamic stretches. Stop if pain occurs.\n\nInput: Suggest a catching drill.\nOutput: Use soft throws, start close, focus on watching the ball, and increase distance gradually.\n\nInput: " + "Suggest a safe 10-minute beginner drill to practise front-foot defence.",
    },
    {
        "task": "Summarize",
        "input": "Summarize in two bullets: A batter should assess the line and length, move their feet early, keep their head steady, and practise consistently. A coach should give specific feedback and avoid changing too many things at once.",
        "few": "Input: Summarize: Warm up first, practise slowly, then increase pace.\nOutput:\n- Warm up before practice.\n- Increase intensity gradually.\n\nInput: Summarize: Watch the ball and keep balance.\nOutput:\n- Track the ball.\n- Maintain balance.\n\nInput: " + "Summarize in two bullets: A batter should assess the line and length, move their feet early, keep their head steady, and practise consistently. A coach should give specific feedback and avoid changing too many things at once.",
    },
    {
        "task": "Give feedback",
        "input": "A fictional batter repeatedly loses balance after attempting a cover drive. Give three constructive coaching suggestions without diagnosing an injury.",
        "few": "Input: A batter swings too early.\nOutput: 1) Track the ball longer. 2) Practise slowly with controlled feeds. 3) Ask a coach to observe timing.\n\nInput: A batter's backlift is inconsistent.\nOutput: 1) Use shadow practice. 2) Repeat a comfortable setup. 3) Review short video clips with a coach.\n\nInput: " + "A fictional batter repeatedly loses balance after attempting a cover drive. Give three constructive coaching suggestions without diagnosing an injury.",
    },
    {
        "task": "Answer with uncertainty",
        "input": "Can a webcam alone prove that a cricket batter will score a century in their next match? Answer briefly and explain the limitation.",
        "few": "Input: Can a video guarantee a player will never miss a shot?\nOutput: No. Video can show technique in the recorded conditions, but it cannot guarantee future performance.\n\nInput: Can one practice session prove long-term improvement?\nOutput: No. Repeated measurements across sessions are needed.\n\nInput: " + "Can a webcam alone prove that a cricket batter will score a century in their next match? Answer briefly and explain the limitation.",
    }
]

techniques = ["Zero-shot", "Few-shot", "Step-by-step", "Role"]
records = []
for task in tasks:
    prompts = {
        "Zero-shot": task["input"],
        "Few-shot": task["few"],
        "Step-by-step": "Answer the task. Give a concise numbered explanation of the main considerations, then a final answer. Do not invent facts.\nTask: " + task["input"],
        "Role": "You are a patient cricket batting coach who gives practical, safe, beginner-friendly advice. Be specific, concise, and state uncertainty when needed.\nTask: " + task["input"],
    }
    for technique in techniques:
        answer = generate(prompts[technique], max_new_tokens=100)
        records.append({"Task":task["task"], "Technique":technique, "Prompt":prompts[technique], "Output":answer,
                        "Word count":len(answer.split())})
        print(f"\n[{task['task']} | {technique}]\n{answer}")

results_df = pd.DataFrame(records)
display(results_df[["Task","Technique","Output","Word count"]])


[Explain a concept | Zero-shot]
Cricket batters should keep their head balanced while playing a straight drive.

[Explain a concept | Few-shot]
Output: Cricket batters should keep their head balanced while playing a straight drive.

[Explain a concept | Step-by-step]
A straight drive is a way to keep your head balanced while playing a straight drive.

[Explain a concept | Role]
If you are a patient cricket batting coach, it is important to keep your head balanced while playing a straight drive.

[Create a drill | Zero-shot]
Using a hammer drill, drill a hammer into a hammer.

[Create a drill | Few-shot]
Output: Use a drill that is easy to use and easy to use.

[Create a drill | Step-by-step]
The first drill is a drill that is designed for beginners. The first drill is a drill that is designed for beginners. The second drill is a drill that is designed for beginners. The third drill is a drill that is designed for beginners. The fourth drill is a drill that is designed for beginners. T

,Task,Technique,Output,Word count
0,Explain a concept,Zero-shot,Cricket batters should keep their head balance...,12
1,Explain a concept,Few-shot,Output: Cricket batters should keep their head...,13
2,Explain a concept,Step-by-step,A straight drive is a way to keep your head ba...,16
3,Explain a concept,Role,"If you are a patient cricket batting coach, it...",21
4,Create a drill,Zero-shot,"Using a hammer drill, drill a hammer into a ha...",10
5,Create a drill,Few-shot,Output: Use a drill that is easy to use and ea...,13
6,Create a drill,Step-by-step,The first drill is a drill that is designed fo...,66
7,Create a drill,Role,"If you are a cricket player, you should be abl...",17
8,Summarize,Zero-shot,Assess the line and length of the batter.,8
9,Summarize,Few-shot,Output: Identify the line and length of the ba...,9


In [ ]:

ratings = results_df[["Task","Technique","Output"]].copy()
for col in ["Correctness (1-5)", "Relevance (1-5)", "Completeness (1-5)", "Clarity (1-5)", "Hallucination observed?"]:
    ratings[col] = ""
display(ratings)

ratings.to_csv("experiment3_prompt_evaluation.csv", index=False)

,Task,Technique,Output,Correctness (1-5),Relevance (1-5),Completeness (1-5),Clarity (1-5),Hallucination observed?
0,Explain a concept,Zero-shot,Cricket batters should keep their head balance...,,,,,
1,Explain a concept,Few-shot,Output: Cricket batters should keep their head...,,,,,
2,Explain a concept,Step-by-step,A straight drive is a way to keep your head ba...,,,,,
3,Explain a concept,Role,"If you are a patient cricket batting coach, it...",,,,,
4,Create a drill,Zero-shot,"Using a hammer drill, drill a hammer into a ha...",,,,,
5,Create a drill,Few-shot,Output: Use a drill that is easy to use and ea...,,,,,
6,Create a drill,Step-by-step,The first drill is a drill that is designed fo...,,,,,
7,Create a drill,Role,"If you are a cricket player, you should be abl...",,,,,
8,Summarize,Zero-shot,Assess the line and length of the batter.,,,,,
9,Summarize,Few-shot,Output: Identify the line and length of the ba...,,,,,
